# pixel classification and analysis on all images
a notebook to execute your classification and analysis

in Colab, we need to connect to a GPU

on the top right, go to change runtime type and select T4 GPU

In [ ]:
pip install readlif

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

In [ ]:
pip install openpyxl

It is normal that in colab, you might neeed to restart the session when you attempt to pip install apoc.... Simply confirm restart of the session and start from the first executable cell

In [ ]:
!pip install apoc --no-deps
!pip install "scikit-learn" "pyclesperanto-prototype" "pandas"
!pip install "numpy==2.4.4"

In [ ]:
from matplotlib import pyplot as plt
import apoc
from skimage.io import imread, imsave
import numpy as np
from readlif.reader import LifFile
import tifffile # for reading TIFF metadata
from skimage.measure import label, regionprops # for quantifying objects in binary images
import pandas as pd # for data handling
import os
import tempfile
import zipfile
from pathlib import Path
from PIL import Image
import io
import openpyxl

This time we will be using a .zip file with all of our images and models...

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    filename, raw_bytes = next(iter(uploaded.items()))
    zip_source = Path(filename)
    zip_source.write_bytes(raw_bytes)
    print(f"Path: {zip_source}")
    print(f"Exists: {zip_source.exists()}")          # save to current directory

except ImportError:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    selected = filedialog.askopenfilename(
        title="Select a ZIP file",
        filetypes=[("ZIP archives", "*.zip")]
    )
    root.destroy()

    zip_source = Path(selected.replace("/", "\\"))
    print(f"Path: {zip_source}")
    print(f"Exists: {zip_source.exists()}")

Now we will be somewhat reusing code from before, but we will pack it in functions so that we can perform the entire analysis in the end. 

In [ ]:
CHANNEL_ORDER = ["DAAO", "vGAT", "Cre", "Gphn"]


def _find_zip_member(zip_obj, suffix):
    # Match by tail so it works even if zip has a top-level folder.
    for name in zip_obj.namelist():
        if name.endswith(suffix) and not Path(name).name.startswith("._"):
            return name
    return None


def _load_segmenter(channel, zip_obj, temp_dir, project_root=None, cache=None):
    if cache is None:
        cache = {}

    if channel in cache:
        return cache[channel]

    rel_model = f"training_4_channels/{channel}/models/{channel}_object_model.cl"

    # 1) Prefer local filesystem model if available
    if project_root is not None:
        model_path = Path(project_root) / rel_model
        if model_path.exists():
            seg = apoc.ObjectSegmenter(opencl_filename=str(model_path))
            cache[channel] = seg
            return seg

    # 2) Else load model from zip into temp file
    member = _find_zip_member(zip_obj, rel_model)
    if member is None:
        raise FileNotFoundError(f"Model not found for channel {channel}: {rel_model}")

    model_tmp = Path(temp_dir) / f"{channel}_object_model.cl"
    if not model_tmp.exists():
        model_tmp.write_bytes(zip_obj.read(member))

    seg = apoc.ObjectSegmenter(opencl_filename=str(model_tmp))
    cache[channel] = seg
    return seg


def _to_numpy(array_like):
    """Convert APOC/OpenCL outputs to NumPy arrays for scikit-image."""
    if isinstance(array_like, np.ndarray):
        return array_like
    if hasattr(array_like, "get"):
        return array_like.get()
    return np.asarray(array_like)


def make_masks(channel, array, zip_obj, temp_dir, project_root=None, segmenter_cache=None):
    segmenter = _load_segmenter(
        channel=channel,
        zip_obj=zip_obj,
        temp_dir=temp_dir,
        project_root=project_root,
        cache=segmenter_cache,
    )
    labels = segmenter.predict(array)
    labels = _to_numpy(labels).astype(np.int32, copy=False)
    return labels


def _summarize_gphn(labels, intensity_image):
    props = regionprops(labels, intensity_image=intensity_image)
    n_objects = len(props)
    mean_area_px = float(np.mean([p.area for p in props])) if props else 0.0
    mean_intensity = float(np.mean([p.mean_intensity for p in props])) if props else 0.0
    return n_objects, mean_area_px, mean_intensity


def execute_zip_lif(zip_source, project_root=None):
    results = []
    segmenter_cache = {}

    with zipfile.ZipFile(zip_source, "r") as z, tempfile.TemporaryDirectory() as tmp:
        lif_members = sorted(
            name for name in z.namelist()
            if name.lower().endswith(".lif")
            and not name.startswith("__MACOSX/")
            and not Path(name).name.startswith("._")
        )

        print(f"Found {len(lif_members)} .lif file(s) in zip")

        for lif_member in lif_members:
            # LifFile needs a file path, so materialize only this .lif temporarily.
            lif_tmp = Path(tmp) / Path(lif_member).name
            lif_tmp.write_bytes(z.read(lif_member))

            lif = LifFile(str(lif_tmp))
            for img in lif.get_iter_image():
                channels_array = np.stack([np.array(ch) for ch in img.get_iter_c(t=0, z=0)])

                # LIF scales are in metadata; convert px^2 to um^2 using inverse x/y scales.
                x_scale, y_scale, z_scale, t_scale = img.scale
                if x_scale == 0 or y_scale == 0:
                    raise ValueError(f"Invalid scale metadata in image {img.name}: x_scale={x_scale}, y_scale={y_scale}")
                area_factor_um2 = (1.0 / x_scale) * (1.0 / y_scale)

                channel_images = {
                    channel: _to_numpy(channels_array[idx])
                    for idx, channel in enumerate(CHANNEL_ORDER)
                }

                labels_by_channel = {}
                for channel in CHANNEL_ORDER:
                    labels_by_channel[channel] = make_masks(
                        channel=channel,
                        array=channel_images[channel],
                        zip_obj=z,
                        temp_dir=tmp,
                        project_root=project_root,
                        segmenter_cache=segmenter_cache,
                    )

                gphn_labels = labels_by_channel["Gphn"]
                gphn_intensity_image = channel_images["Gphn"]
                vgat_labels = labels_by_channel["vGAT"]
                daao_labels = labels_by_channel["DAAO"]

                # All gephyrin clusters in this image
                n_gphn, mean_gphn_area_px, mean_gphn_intensity = _summarize_gphn(
                    gphn_labels, gphn_intensity_image
                )

                # Synaptic gephyrin: gephyrin objects overlapping vGAT-positive pixels
                overlap = (gphn_labels > 0) & (vgat_labels > 0)
                overlapping_labels = np.unique(gphn_labels[overlap])
                overlapping_labels = overlapping_labels[overlapping_labels != 0]

                gphn_vgat_positive = np.isin(gphn_labels, overlapping_labels)
                synaptic_gphn_labels = label(gphn_vgat_positive)

                n_syn_gphn, mean_syn_gphn_area_px, mean_syn_gphn_intensity = _summarize_gphn(
                    synaptic_gphn_labels, gphn_intensity_image
                )

                # Total segmented DAAO area converted to um^2
                total_daao_area_px = int(np.count_nonzero(daao_labels > 0))

                results.append({
                    "source_lif": Path(lif_member).name,
                    "image_name": img.name,
                    "#_gephyrin_clusters": int(n_gphn),
                    "#_synaptic_gephyrin_clusters": int(n_syn_gphn),
                    "mean_gephyrin_area_um2": mean_gphn_area_px * area_factor_um2,
                    "mean_synaptic_gephyrin_area_um2": mean_syn_gphn_area_px * area_factor_um2,
                    "mean_gephyrin_intensity": mean_gphn_intensity,
                    "mean_synaptic_gephyrin_intensity": mean_syn_gphn_intensity,
                    "total_DAAO_area_um2": total_daao_area_px * area_factor_um2,
                    "pixel_in_x": 1/x_scale,
                    "pixel_in_y": 1/y_scale,
                })

    return pd.DataFrame(results)

Finally we can run the entire analysis. 

In [ ]:
# Build one-row-per-image results table
final_df = execute_zip_lif(zip_source, project_root=Path(".").resolve())

# Show the final per-image table (areas in um^2)
display(final_df)


We can quickly summarize by condition...

In [ ]:
# Optional compact summary by source .lif
summary_by_lif = (
    final_df.groupby("source_lif", as_index=False)
    .agg(
        images=("image_name", "nunique"),
        mean_gephyrin_clusters=("#_gephyrin_clusters", "mean"),
        mean_synaptic_gephyrin_clusters=("#_synaptic_gephyrin_clusters", "mean"),
        mean_gephyrin_area_um2=("mean_gephyrin_area_um2", "mean"),
        mean_synaptic_gephyrin_area_um2=("mean_synaptic_gephyrin_area_um2", "mean"),
        mean_gephyrin_intensity=("mean_gephyrin_intensity", "mean"),
        mean_synaptic_gephyrin_intensity=("mean_synaptic_gephyrin_intensity", "mean"),
        mean_total_DAAO_area_um2=("total_DAAO_area_um2", "mean"),
    )
    .sort_values("source_lif")
    .reset_index(drop=True)
)

display(summary_by_lif)

Finally, we will save the measured parameters.

In [ ]:
try:
    from google.colab import files
    final_df.to_excel('final_df.xlsx', index=False)
    print(final_df.head())
    files.download('final_df.xlsx')

except ImportError:
    final_df.to_excel('final_df.xlsx', index=False)
    print(final_df.head())
 